# Demo 10. Root-Finding and the Cobweb Diagram

**Standard 9** (nonlinear equations), Week 7.

Bisection, Newton, secant, and fixed-point iteration all solve $f(x)=0$, but at different speeds. This demo measures the order of convergence of each and uses the cobweb diagram to show when fixed-point iteration converges.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Orders of convergence

A sequence converges with order $p$ if $|e_{k+1}| \le C|e_k|^p$ for $e_k = x_k - x^\*$. Bisection halves the bracket each step, so $|e_{k+1}| \approx \tfrac12 |e_k|$ (order 1). Newton's method satisfies $|e_{k+1}| \approx \tfrac{|f''(x^\*)|}{2|f'(x^\*)|}|e_k|^2$ (order 2) at a simple root. The secant method has order $(1+\sqrt5)/2 \approx 1.618$. Fixed-point iteration $x_{k+1}=g(x_k)$ converges locally when $|g'(x^\*)|<1$, with error ratio approaching $|g'(x^\*)|$.

In [ ]:
# ---------------------------------------------------------------------------
# Four scalar root-finders, each returning its full iterate history
# ---------------------------------------------------------------------------
# Returning the history lets us measure the ORDER of convergence: how fast the
# error e_k = x_k - x* shrinks. Bisection is linear (halves the bracket each
# step), Newton is quadratic, and the secant method has order ~1.618.

def bisection(f, a, b, n_iter):
    """Bisection on a sign-changing bracket [a, b]. Linear convergence."""
    xs = []
    for _ in range(n_iter):
        m = 0.5 * (a + b)
        xs.append(m)
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
    return np.array(xs)

def newton(f, fp, x0, n_iter):
    """Newton's method x <- x - f(x)/f'(x). Quadratic convergence near a simple
    root, provided f'(x) is not zero there."""
    xs = [x0]
    x = x0
    for _ in range(n_iter):
        x = x - f(x) / fp(x)
        xs.append(x)
    return np.array(xs)

def secant(f, x0, x1, n_iter):
    """Secant method: Newton with the derivative replaced by a finite difference
    from the two previous iterates. Order ~1.618, no derivative required."""
    xs = [x0, x1]
    for _ in range(n_iter):
        f0, f1 = f(xs[-2]), f(xs[-1])
        denom = f1 - f0
        if denom == 0:                       # converged: f-values coincide
            break
        xs.append(xs[-1] - f1 * (xs[-1] - xs[-2]) / denom)
    return np.array(xs)

def fixed_point(g, x0, n_iter):
    """Fixed-point iteration x <- g(x). Converges to a fixed point x*=g(x*) when
    |g'(x*)| < 1; the cobweb plot below shows why."""
    xs = [x0]
    for _ in range(n_iter):
        xs.append(g(xs[-1]))
    return np.array(xs)

In [ ]:
# ---------------------------------------------------------------------------
# Measuring the order of convergence from the error sequence
# ---------------------------------------------------------------------------
# If |e_{k+1}| ~ C |e_k|^p, then p is estimated by successive ratios of logs:
#     p ~ log(|e_{k+1}|/|e_k|) / log(|e_k|/|e_{k-1}|).

def estimated_order(xs, x_star):
    """Estimate the convergence order p from an iterate history."""
    e = np.abs(np.asarray(xs, float) - x_star)
    e = e[e > 0]                             # drop exact-zero errors
    if len(e) < 4:
        return float("nan")
    p = []
    for k in range(1, len(e) - 1):
        num = np.log(e[k+1] / e[k])
        den = np.log(e[k] / e[k-1])
        if den != 0:
            p.append(num / den)
    return np.median(p) if p else float("nan")

In [ ]:
# ---------------------------------------------------------------------------
# Compare error decay and estimate each method's order
# ---------------------------------------------------------------------------
# Test problem: f(x) = x^3 - x - 2, with a single real root near 1.5214.
def f(x):  return x**3 - x - 2.0
def fp(x): return 3.0*x**2 - 1.0
X_STAR = 1.5213797068045675            # high-accuracy root for error measurement

def show_convergence(n_iter=8):
    """Run bisection, Newton, and secant; plot error vs iteration and print the
    estimated order of each."""
    xb = bisection(f, 1.0, 2.0, 40)
    xn = newton(f, fp, 1.5, n_iter)
    xsq = secant(f, 1.0, 2.0, n_iter)

    plt.figure()
    plt.semilogy(np.abs(xb - X_STAR) + 1e-18, "b.-", label="bisection")
    plt.semilogy(np.abs(xn - X_STAR) + 1e-18, "r.-", label="Newton")
    plt.semilogy(np.abs(xsq - X_STAR) + 1e-18, "g.-", label="secant")
    plt.xlabel("iteration"); plt.ylabel("|x_k - x*|")
    plt.title("Error decay of three root-finders"); plt.legend(); plt.show()

    print("bisection : bracket halves each step, linear with rate 1/2 by construction")
    print(f"estimated order  Newton    : {estimated_order(xn, X_STAR):.2f}  (theory 2)")
    print(f"estimated order  secant    : {estimated_order(xsq, X_STAR):.2f}  (theory 1.62)")

show_convergence(8)

In [ ]:
# ---------------------------------------------------------------------------
# Cobweb diagram for fixed-point iteration x <- g(x)
# ---------------------------------------------------------------------------
# The cobweb traces x0 -> g(x0) up to the curve, across to the line y=x, and
# repeats. It spirals or staircases INTO the fixed point when |g'(x*)| < 1 and
# AWAY from it when |g'(x*)| > 1.
GFUNS = {
    "g(x) = cos(x)          (converges)": (np.cos, 0.0, 1.5),
    "g(x) = (x+2)^(1/3)     (converges)": (lambda x: np.cbrt(x + 2.0), 0.0, 2.0),
    "g(x) = x^2 - 0.2       (diverges near upper root)": (lambda x: x**2 - 0.2, 0.0, 1.3),
}

def show_cobweb(choice=list(GFUNS)[0], x0=0.5, n_iter=12):
    """Draw the cobweb diagram for the selected fixed-point map."""
    g, a, b = GFUNS[choice]
    xs = fixed_point(g, x0, n_iter)

    t = np.linspace(a, b, 400)
    plt.figure()
    plt.plot(t, g(t), "b-", lw=2, label="y = g(x)")
    plt.plot(t, t, "k--", lw=1, label="y = x")
    # cobweb path: (x_k, x_k) -> (x_k, g(x_k)) -> (x_{k+1}, x_{k+1})
    for k in range(len(xs) - 1):
        plt.plot([xs[k], xs[k]], [xs[k], xs[k+1]], "r-", lw=1)
        plt.plot([xs[k], xs[k+1]], [xs[k+1], xs[k+1]], "r-", lw=1)
    plt.plot(xs[0], g(xs[0]), "go", label="start")
    plt.xlabel("x"); plt.ylabel("g(x)"); plt.legend()
    plt.title("Fixed-point iteration, cobweb diagram"); plt.show()
    print("last three iterates:", xs[-3:])

show_cobweb()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive controls
# ---------------------------------------------------------------------------
interact(show_convergence, n_iter=FloatSlider(value=8, min=3, max=15, step=1, description="Newton/secant iters"));
interact(
    show_cobweb,
    choice=Dropdown(options=list(GFUNS), description="map g"),
    x0=FloatSlider(value=0.5, min=0.0, max=1.3, step=0.05, description="x0"),
    n_iter=FloatSlider(value=12, min=2, max=30, step=1, description="# iters"),
);

## Summary

- Bisection is guaranteed on a sign-changing bracket but only linear.
- Newton is quadratic near a simple root but needs the derivative and a good start.
- Secant achieves order $\approx 1.618$ without a derivative.
- Fixed-point iteration converges when $|g'(x^\*)|<1$; the cobweb staircases in when the slope is shallow and spirals out when it is steep.